In [27]:
import pandas as pd
import ast
import re
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss
import networkx as nx
from rapidfuzz import process, fuzz
from collections import Counter
import ast
import pyarrow
from ingredient_parser import parse_ingredient
from fractions import Fraction

aliases

In [7]:
UNIT_ALIASES = {
    "t": "teaspoon", "t.": "teaspoon", "tsp": "teaspoon", "tsp.": "teaspoon",
    "teaspoon": "teaspoon", "teaspoons": "teaspoon",

    "T": "tablespoon", "T.": "tablespoon", "tbsp": "tablespoon", "tbsp.": "tablespoon",
    "tbs": "tablespoon", "tbs.": "tablespoon", "tablespoon": "tablespoon",
    "tablespoons": "tablespoon",

    "c": "cup", "c.": "cup", "cup": "cup", "cups": "cup",

    "oz": "ounce", "oz.": "ounce", "ounce": "ounce", "ounces": "ounce",

    "lb": "pound", "lb.": "pound", "lbs": "pound", "lbs.": "pound",
    "pound": "pound", "pounds": "pound",

    "g": "gram", "g.": "gram", "gram": "gram", "grams": "gram",

    "kg": "kilogram", "kg.": "kilogram", "kilogram": "kilogram",
    "kilograms": "kilogram",

    "ml": "milliliter", "ml.": "milliliter", "mL": "milliliter",
    "milliliter": "milliliter", "milliliters": "milliliter",

    "l": "liter", "L": "liter", "liter": "liter", "liters": "liter",

    "pinch": "pinch", "pinches": "pinch",
    "dash": "dash", "dashes": "dash",
    "clove": "clove", "cloves": "clove",
    "slice": "slice", "slices": "slice",
    "can": "can", "cans": "can",
    "package": "package", "packages": "package",
    "pkg": "package", "pkg.": "package",
}

BASE_PATH = "./Data"

load state

In [8]:
recipes_df = pd.read_parquet(f"{BASE_PATH}/recipes_trimmed_keep_longest.parquet")
ingredient_matches = pd.read_parquet(f"{BASE_PATH}/accepted_ingredient_matches_v1.parquet")
ingredient_candidates = pd.read_parquet(f"{BASE_PATH}/ingredient_food_candidates_v1.parquet")

embeddings = np.load(f"{BASE_PATH}/recipe_embeddings_fp16.npy")
index = faiss.read_index(f"{BASE_PATH}/recipe_index.faiss")

In [9]:
recipes_df["ingredients"].iloc[0]

array(['1 c. firmly packed brown sugar', '1/2 c. evaporated milk',
       '1/2 tsp. vanilla', '1/2 c. broken nuts (pecans)',
       '2 Tbsp. butter or margarine',
       '3 1/2 c. bite size shredded rice biscuits'], dtype=object)

create safe copy

In [10]:
recipes = recipes_df.copy(deep=True)

new parsed column

In [11]:
recipes["ingredients_parsed"] = recipes["ingredients"]

flatten for analysis

In [12]:
all_ingredients = [
    ing
    for ing_list in recipes["ingredients_parsed"]
    if isinstance(ing_list, (list, np.ndarray))
    for ing in ing_list
]

In [13]:
len(all_ingredients), all_ingredients[:10]

(16173715,
 ['1 c. firmly packed brown sugar',
  '1/2 c. evaporated milk',
  '1/2 tsp. vanilla',
  '1/2 c. broken nuts (pecans)',
  '2 Tbsp. butter or margarine',
  '3 1/2 c. bite size shredded rice biscuits',
  '1 small jar chipped beef, cut up',
  '4 boned chicken breasts',
  '1 can cream of mushroom soup',
  '1 carton sour cream'])

Now testing a ingredient unit parser based on these raw text keys alone. This will not be the final product

In [14]:
def clean_unit_token(token):
    if token is None:
        return None

    token = token.strip()

    if token in ["T", "T.", "L"]:
        return token

    return token.lower().replace(".", "")


def extract_possible_unit(line):
    pattern = r"^\s*(\d+\s+\d+/\d+|\d+/\d+|\d+(?:\.\d+)?)\s+([A-Za-z.]+)"
    match = re.match(pattern, str(line))

    if not match:
        return None

    return clean_unit_token(match.group(2))


def find_unknown_units(ingredient_lines):
    possible_units = []

    for line in ingredient_lines:
        unit = extract_possible_unit(line)
        if unit:
            possible_units.append(unit)

    counts = Counter(possible_units)

    unknown = {
        unit: count
        for unit, count in counts.items()
        if unit not in UNIT_ALIASES
    }

    return dict(sorted(unknown.items(), key=lambda x: x[1], reverse=True))

In [15]:
unknown_units = find_unknown_units(all_ingredients)

list(unknown_units.items())[:20]

[('large', 319767),
 ('eggs', 191992),
 ('medium', 185730),
 ('small', 160638),
 ('egg', 142224),
 ('garlic', 111620),
 ('whole', 108879),
 ('to', 107269),
 ('green', 59922),
 ('onion', 59182),
 ('stick', 55116),
 ('x', 44830),
 ('red', 41653),
 ('bunch', 35770),
 ('bay', 34343),
 ('box', 31967),
 ('each', 29480),
 ('qt', 29478),
 ('chicken', 29098),
 ('lemon', 28711)]

In [16]:
len(unknown_units)

10175

In [17]:
import nltk
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('punkt')

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\winni\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\winni\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.


True

In [18]:
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\winni\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping taggers\averaged_perceptron_tagger.zip.


True

A good start, but I would like to run the remaining and sanity check the rest as well with an LLM solution

starting with a sample set

In [19]:
sample_ingredients = all_ingredients[:10000]

In [20]:
def parse_with_status(line):
    try:
        parsed = parse_ingredient(str(line))

        has_amount = bool(parsed.amount)
        has_name = bool(parsed.name)

        if has_amount and has_name:
            status = "parsed"
        elif has_name and not has_amount:
            status = "missing_amount"
        else:
            status = "failed"

        return {
            "ingredient_raw": line,
            "status": status,
            "parsed": parsed,
            "error": None
        }

    except Exception as e:
        return {
            "ingredient_raw": line,
            "status": "error",
            "parsed": None,
            "error": str(e)
        }

parse

In [21]:
parsed_results = [parse_with_status(x) for x in sample_ingredients]

audit table of parse

In [22]:
parse_audit = pd.DataFrame([
    {
        "ingredient_raw": r["ingredient_raw"],
        "status": r["status"],
        "parsed": str(r["parsed"]),
        "error": r["error"]
    }
    for r in parsed_results
])

explore

In [23]:
parse_audit["status"].value_counts()

status
parsed            9315
missing_amount     675
failed              10
Name: count, dtype: int64

In [24]:
for x in sample_ingredients[:5]:
    try:
        print(parse_ingredient(str(x)))
    except Exception as e:
        print(type(e))
        print(e)
        break

ParsedIngredient(name=[IngredientText(text='brown sugar', confidence=0.98461, starting_index=5)], size=None, amount=[IngredientAmount(quantity=Fraction(1, 1), quantity_max=Fraction(1, 1), unit=<Unit('cup')>, text='1 c', confidence=0.999485, starting_index=0, APPROXIMATE=False, SINGULAR=False, RANGE=False, MULTIPLIER=False, PREPARED_INGREDIENT=False)], preparation=None, comment=IngredientText(text='firmly packed', confidence=0.981001, starting_index=3), purpose=None, foundation_foods=[], sentence='1 c. firmly packed brown sugar')
ParsedIngredient(name=[IngredientText(text='evaporated milk', confidence=0.942382, starting_index=3)], size=None, amount=[IngredientAmount(quantity=Fraction(1, 2), quantity_max=Fraction(1, 2), unit=<Unit('cup')>, text='1/2 c', confidence=0.999456, starting_index=0, APPROXIMATE=False, SINGULAR=False, RANGE=False, MULTIPLIER=False, PREPARED_INGREDIENT=False)], preparation=None, comment=None, purpose=None, foundation_foods=[], sentence='1/2 c. evaporated milk')
Pa

## Hybrid Ingredient Parsing Pipeline

This section uses a hybrid parsing approach:
1. A deterministic parser handles high-confidence ingredient lines.
2. Ambiguous or failed parses are flagged for AI/ML fallback.
3. AI outputs are validated before being accepted.

de-dupe

In [25]:
unique_ingredients = (
    pd.Series(all_ingredients)
    .dropna()
    .astype(str)
    .str.strip()
    .drop_duplicates()
    .reset_index(drop=True)
)

len(unique_ingredients)

4361045

take sample

In [26]:
sample_unique_ingredients = unique_ingredients.sample(
    n=10000,
    random_state=42
).reset_index(drop=True)

sample_unique_ingredients.head()

0        3/4 cup fresh lime juice, about 3 large limes
1             1 Tbsp. Popcorn oil,or possibly veg. oil
2    1/2 cup seeded and sliced orange mini bell pepper
3                         1/2 cup thinly-sliced celery
4          2 jalapeno peppers, chopped (without seeds)
dtype: object

new aliases

In [28]:
UNIT_ALIASES = {
    # volume
    "cup": "cup", "cups": "cup", "c": "cup", "c.": "cup",
    "tablespoon": "tablespoon", "tablespoons": "tablespoon",
    "tbsp": "tablespoon", "tbsp.": "tablespoon", "tbs": "tablespoon", "T": "tablespoon",
    "teaspoon": "teaspoon", "teaspoons": "teaspoon",
    "tsp": "teaspoon", "tsp.": "teaspoon", "t": "teaspoon",

    # weight
    "oz": "ounce", "oz.": "ounce", "ounce": "ounce", "ounces": "ounce",
    "lb": "pound", "lb.": "pound", "lbs": "pound", "pound": "pound", "pounds": "pound",
    "g": "gram", "gram": "gram", "grams": "gram",
    "kg": "kilogram", "kilogram": "kilogram", "kilograms": "kilogram",

    # liquid
    "ml": "milliliter", "milliliter": "milliliter", "milliliters": "milliliter",
    "l": "liter", "liter": "liter", "liters": "liter",
    "pint": "pint", "pints": "pint", "pt": "pint",
    "quart": "quart", "quarts": "quart", "qt": "quart",
    "gallon": "gallon", "gallons": "gallon", "gal": "gallon",

    # count/container
    "can": "can", "cans": "can",
    "package": "package", "packages": "package", "pkg": "package",
    "box": "box", "boxes": "box",
    "jar": "jar", "jars": "jar",
    "bottle": "bottle", "bottles": "bottle",
    "stick": "stick", "sticks": "stick",
    "slice": "slice", "slices": "slice",
    "piece": "piece", "pieces": "piece",
    "clove": "clove", "cloves": "clove",
    "bunch": "bunch", "bunches": "bunch",
    "head": "head", "heads": "head",
    "stalk": "stalk", "stalks": "stalk",
    "sprig": "sprig", "sprigs": "sprig",

    # fallback count
    "each": "each", "ea": "each",
}

SIZE_WORDS = {
    "small", "medium", "large", "jumbo",
    "extra-large", "extra", "whole"
}

PREP_WORDS = {
    "chopped", "diced", "minced", "sliced", "crushed", "grated",
    "shredded", "beaten", "melted", "softened", "drained",
    "rinsed", "peeled", "cooked", "uncooked", "fresh", "frozen",
    "dry", "dried", "ground"
}

TEXT_AMOUNTS = {
    "a": 1,
    "an": 1,
    "one": 1,
    "two": 2,
    "three": 3,
    "four": 4,
}

quantity parsing

In [29]:
def parse_quantity(qty_text):
    if qty_text is None:
        return None

    qty_text = str(qty_text).strip().lower()

    if qty_text in TEXT_AMOUNTS:
        return float(TEXT_AMOUNTS[qty_text])

    try:
        # handles "1 1/2"
        if " " in qty_text:
            parts = qty_text.split()
            return float(Fraction(parts[0])) + float(Fraction(parts[1]))

        # handles "1/2" and "2"
        return float(Fraction(qty_text))

    except Exception:
        return None

test parse

In [30]:
for q in ["1", "1/2", "1 1/2", "2.5", "a", "three"]:
    print(q, "->", parse_quantity(q))

1 -> 1.0
1/2 -> 0.5
1 1/2 -> 1.5
2.5 -> 2.5
a -> 1.0
three -> 3.0


first pass parser

In [31]:
def normalize_unit(unit_raw):
    if unit_raw is None:
        return None

    unit_clean = str(unit_raw).strip()

    # preserve capital T as tablespoon if needed
    if unit_clean == "T":
        return "tablespoon"

    unit_clean = unit_clean.lower().replace(".", "")
    return UNIT_ALIASES.get(unit_clean)


def simple_rule_parse(line):
    line = str(line).strip()

    # Remove extra spaces
    line = re.sub(r"\s+", " ", line)

    # Pattern for:
    # 1 cup sugar
    # 1/2 tsp salt
    # 1 1/2 cups flour
    # a pinch salt
    pattern = r"^((?:\d+\s+\d+/\d+)|(?:\d+/\d+)|(?:\d+(?:\.\d+)?)|a|an|one|two|three|four)\s+([A-Za-z.]+)?\s*(.*)$"

    match = re.match(pattern, line, flags=re.IGNORECASE)

    if not match:
        return {
            "ingredient_raw": line,
            "quantity": None,
            "unit": None,
            "food_name": line,
            "size": None,
            "preparation": None,
            "parse_source": "rule",
            "parse_status": "no_quantity",
            "confidence": 0.35
        }

    qty_raw, possible_unit, rest = match.groups()
    quantity = parse_quantity(qty_raw)

    possible_unit_clean = possible_unit.lower().replace(".", "") if possible_unit else None
    unit = normalize_unit(possible_unit)

    size = None

    # Case: "2 large eggs"
    if unit is None and possible_unit_clean in SIZE_WORDS:
        size = possible_unit_clean
        unit = "each"
        food_name = rest.strip()
        status = "parsed_size_as_count"
        confidence = 0.80

    # Case: "4 eggs"
    elif unit is None:
        unit = "each"
        food_name = f"{possible_unit or ''} {rest}".strip()
        status = "quantity_no_known_unit"
        confidence = 0.60

    # Case: "1 cup sugar"
    else:
        food_name = rest.strip()
        status = "parsed"
        confidence = 0.90

    # Extract simple preparation words from food name
    tokens = food_name.lower().replace(",", "").split()
    prep_found = [t for t in tokens if t in PREP_WORDS]

    preparation = ", ".join(sorted(set(prep_found))) if prep_found else None

    # Remove prep words from food name guess
    clean_food_tokens = [t for t in food_name.split() if t.lower().strip(",") not in PREP_WORDS]
    clean_food_name = " ".join(clean_food_tokens).strip(" ,")

    if clean_food_name == "":
        clean_food_name = food_name

    return {
        "ingredient_raw": line,
        "quantity": quantity,
        "unit": unit,
        "food_name": clean_food_name,
        "size": size,
        "preparation": preparation,
        "parse_source": "rule",
        "parse_status": status,
        "confidence": confidence
    }

quick hand picked set test

In [32]:
test_lines = [
    "1 cup sugar",
    "1/2 tsp salt",
    "2 tbsp olive oil",
    "3 large eggs, beaten",
    "1 medium onion, diced",
    "salt to taste",
    "juice of 1 lemon",
    "1 can cream of mushroom soup",
    "a pinch of pepper",
    "2 cloves garlic, minced"
]

pd.DataFrame([simple_rule_parse(x) for x in test_lines])

,ingredient_raw,quantity,unit,food_name,size,preparation,parse_source,parse_status,confidence
0,1 cup sugar,1.0,cup,sugar,None,None,rule,parsed,0.90
1,1/2 tsp salt,0.5,teaspoon,salt,None,None,rule,parsed,0.90
2,2 tbsp olive oil,2.0,tablespoon,olive oil,None,None,rule,parsed,0.90
3,"3 large eggs, beaten",3.0,each,eggs,large,beaten,rule,parsed_size_as_count,0.80
4,"1 medium onion, diced",1.0,each,onion,medium,diced,rule,parsed_size_as_count,0.80
5,salt to taste,NaN,None,salt to taste,None,None,rule,no_quantity,0.35
6,juice of 1 lemon,NaN,None,juice of 1 lemon,None,None,rule,no_quantity,0.35
7,1 can cream of mushroom soup,1.0,can,cream of mushroom soup,None,None,rule,parsed,0.90
8,a pinch of pepper,1.0,each,pinch of pepper,None,None,rule,quantity_no_known_unit,0.60
9,"2 cloves garlic, minced",2.0,clove,garlic,None,minced,rule,parsed,0.90


parse sample

In [33]:
rule_parsed_sample = pd.DataFrame(
    [simple_rule_parse(x) for x in sample_unique_ingredients]
)

rule_parsed_sample.head()

,ingredient_raw,quantity,unit,food_name,size,preparation,parse_source,parse_status,confidence
0,"3/4 cup fresh lime juice, about 3 large limes",0.75,cup,"lime juice, about 3 large limes",None,fresh,rule,parsed,0.9
1,"1 Tbsp. Popcorn oil,or possibly veg. oil",1.00,tablespoon,"Popcorn oil,or possibly veg. oil",None,None,rule,parsed,0.9
2,1/2 cup seeded and sliced orange mini bell pepper,0.50,cup,seeded and orange mini bell pepper,None,sliced,rule,parsed,0.9
3,1/2 cup thinly-sliced celery,0.50,cup,thinly-sliced celery,None,None,rule,parsed,0.9
4,"2 jalapeno peppers, chopped (without seeds)",2.00,each,"jalapeno peppers, (without seeds)",None,chopped,rule,quantity_no_known_unit,0.6


inspect statuses

In [34]:
rule_parsed_sample["parse_status"].value_counts()

parse_status
parsed                    5841
quantity_no_known_unit    2466
no_quantity               1100
parsed_size_as_count       593
Name: count, dtype: int64

distribution of confidence

In [35]:
rule_parsed_sample["confidence"].describe()

count    10000.000000
mean         0.759590
std          0.190659
min          0.350000
25%          0.600000
50%          0.900000
75%          0.900000
max          0.900000
Name: confidence, dtype: float64

weak parses

In [36]:
rule_parsed_sample.sort_values("confidence").head(50)

,ingredient_raw,quantity,unit,food_name,size,preparation,parse_source,parse_status,confidence
5149,approx 2tbsp milk,NaN,None,approx 2tbsp milk,None,None,rule,no_quantity,0.35
894,or defrosted if store bought,NaN,None,or defrosted if store bought,None,None,rule,no_quantity,0.35
4682,ground chili peppers to taste (optional),NaN,None,ground chili peppers to taste (optional),None,None,rule,no_quantity,0.35
891,"fresh oregano,",NaN,None,"fresh oregano,",None,None,rule,no_quantity,0.35
7509,Pillsbury Grands refrigerated buttermilk biscu...,NaN,None,Pillsbury Grands refrigerated buttermilk biscu...,None,None,rule,no_quantity,0.35
8987,"Shrimp Hushpuppies, for serving, recipe follows",NaN,None,"Shrimp Hushpuppies, for serving, recipe follows",None,None,rule,no_quantity,0.35
6909,Pinch of ground paprika,NaN,None,Pinch of ground paprika,None,None,rule,no_quantity,0.35
4683,-14 heaping spoon fulls of sugar or sweet and low,NaN,None,-14 heaping spoon fulls of sugar or sweet and low,None,None,rule,no_quantity,0.35
8984,Mustard & Maple Marinade,NaN,None,Mustard & Maple Marinade,None,None,rule,no_quantity,0.35
884,500g diced Lamb (a little fat on the meat is f...,NaN,None,500g diced Lamb (a little fat on the meat is f...,None,None,rule,no_quantity,0.35


inspect by status

In [37]:
rule_parsed_sample[
    rule_parsed_sample["parse_status"].isin(["no_quantity", "quantity_no_known_unit"])
].sample(50, random_state=42)

,ingredient_raw,quantity,unit,food_name,size,preparation,parse_source,parse_status,confidence
2486,"1 14 cups dried apples, coarsely chopped",1.0,each,"14 cups apples, coarsely",None,"chopped, dried",rule,quantity_no_known_unit,0.60
9642,"1 - 5 ounce can sliced water chestnuts, cut in...",1.0,each,"- 5 ounce can water chestnuts, cut in quarters",None,sliced,rule,quantity_no_known_unit,0.60
902,"3 zucchini, about 1 1/2 pounds",3.0,each,"zucchini , about 1 1/2 pounds",None,None,rule,quantity_no_known_unit,0.60
2657,1 1/2 to 2 lb. ground chuck or sausage,1.5,each,to 2 lb. chuck or sausage,None,ground,rule,quantity_no_known_unit,0.60
5188,1 bag stick pretzels (12 oz.),1.0,each,bag stick pretzels (12 oz.),None,None,rule,quantity_no_known_unit,0.60
8809,1 LG Spanish Onion(Slice Thin),1.0,each,LG Spanish Onion(Slice Thin),None,None,rule,quantity_no_known_unit,0.60
2791,"3 garlic cloves, minced (or 1 1/2 teaspoons mi...",3.0,each,"garlic cloves, (or 1 1/2 teaspoons minced)",None,minced,rule,quantity_no_known_unit,0.60
185,10 or 11 medium potatoes (approximately 4 c. a...,10.0,each,or 11 medium potatoes (approximately 4 c. afte...,None,None,rule,quantity_no_known_unit,0.60
6742,"4 x Green chillies, minced fine",4.0,each,"x Green chillies, fine",None,minced,rule,quantity_no_known_unit,0.60
9211,1 scotch pancake,1.0,each,scotch pancake,None,None,rule,quantity_no_known_unit,0.60


define AI fallback confidence level

In [ ]:
needs_ai = rule_parsed_sample[
    (rule_parsed_sample["confidence"] < 0.80) |
    (rule_parsed_sample["parse_status"].isin(["no_quantity", "quantity_no_known_unit"]))
].copy()

len(needs_ai)

In [ ]:
needs_ai[["ingredient_raw", "parse_status", "food_name", "confidence"]].head(100)